# Mimicore cloud voice training

Trains your **Mimicore** voice on a free Google GPU, using **your own** Google account and Drive.

1. Press **▶** on the cell below (or *Runtime > Run all*).
2. Click **Run anyway**, then allow **Google Drive** access.
3. Keep this tab open. The Mimicore app shows the progress and installs the voice when it's done.

**Your data:** the recordings are in your Drive (`Mimicore Cloud Training` folder) and this temporary Colab machine,
which Google wipes when the session ends. Nothing is sent to Ribanix Labs. Recordings and checkpoints are deleted from
your Drive when the voice is installed, unless you chose to keep them.


In [ ]:
#@title ▶ Train my Mimicore voice
#@markdown Press the ▶ button on the left. Then click **Run anyway** and allow **Google Drive** access.<br>
#@markdown Keep this tab open until it says **Done** (about 1–2 hours for Good quality). Mimicore shows the progress too.<br>
#@markdown <br>**Privacy:** this notebook only reads and writes the `Mimicore Cloud Training` folder in *your* Google Drive.
#@markdown Your recordings never go to Ribanix Labs. The code is open: https://github.com/ribanixlabs/mimicore-app/tree/main/colab
import json, os, re, shutil, subprocess, sys, tarfile, threading, time, traceback, urllib.request
from pathlib import Path

ROOT_NAME = "Mimicore Cloud Training"
REPO_RAW = "https://raw.githubusercontent.com/ribanixlabs/mimicore-app/main/colab"
APPLIO_COMMIT = "ee15d4eb0d5c97106f76de57ffb1addd2c612002"
TITAN_URL = "https://huggingface.co/blaise-tk/TITAN/resolve/main/models/medium/48k/pretrained/{}-f048k-TITAN-Medium.pth"
HOME = Path("/content/mimicore")
APPLIO = HOME / "applio"
TOOLS = HOME / "tools"
WORK = Path("/content/work")


def banner(text, color="#7c6cf0"):
    from IPython.display import HTML, display
    display(HTML(f'<div style="padding:10px 14px;border-radius:10px;background:{color};color:#fff;font:600 15px system-ui">{text}</div>'))


def sh(cmd, cwd=None, quiet=True):
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), capture_output=quiet, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{cmd if isinstance(cmd, str) else ' '.join(map(str, cmd))} failed:\n{(r.stdout or '')[-1500:]}{(r.stderr or '')[-1500:]}")
    return r.stdout


class Job:
    """One training job: a folder in My Drive/Mimicore Cloud Training created by the Mimicore app or website."""

    def __init__(self, folder: Path):
        self.folder = folder
        self.spec = json.loads((folder / "job.json").read_text())
        self.status_file = folder / "status.json"
        self.status = {}
        try:
            self.status = json.loads(self.status_file.read_text() or "{}")
        except Exception:
            pass

    def write(self, **kw):
        # Update the file in place (same Drive file id) so the app, which can only see files it created, can read it.
        self.status.update(kw, updated=int(time.time()))
        data = json.dumps(self.status).encode()
        with open(self.status_file, "r+b") as f:
            f.seek(0)
            f.write(data)
            f.truncate()
            f.flush()
            os.fsync(f.fileno())

    def cancelled(self):
        try:
            return bool(json.loads((self.folder / "job.json").read_text()).get("cancel"))
        except Exception:
            return False


def find_job():
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / ROOT_NAME
    if not root.exists():
        raise SystemExit("No training job found. Start one in the Mimicore app (Voices > Train a voice > Train in the cloud) "
                         "or on the website, then run this again.")
    jobs = []
    for jf in root.glob("*/job.json"):
        j = Job(jf.parent)
        if j.status.get("state") not in ("done", "installed") and not j.cancelled():
            jobs.append(j)
    if not jobs:
        raise SystemExit("No waiting training job. Start one in the Mimicore app or on the website, then run this again.")
    return max(jobs, key=lambda j: j.spec.get("created", 0))


def setup(job):
    job.write(state="setup", label="Preparing the cloud GPU (about 5–10 minutes)")
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
    if gpu.returncode != 0:
        raise SystemExit("This session has no GPU. Click Runtime > Change runtime type > T4 GPU > Save, then press ▶ again. "
                         "If Colab says no GPU is available, try again later: free GPUs are limited at busy times.")
    name, mem = [x.strip() for x in gpu.stdout.strip().splitlines()[0].split(",")]
    vram = float(mem) / 1024
    print(f"GPU: {name} ({vram:.0f} GB)")
    if not (APPLIO / "core.py").exists():
        HOME.mkdir(parents=True, exist_ok=True)
        print("Downloading the training engine (Applio)...")
        sh(["git", "clone", "-q", "https://github.com/IAHispano/Applio", str(APPLIO)])
        sh(["git", "-c", "advice.detachedHead=false", "checkout", "-q", APPLIO_COMMIT], cwd=APPLIO)
    TOOLS.mkdir(parents=True, exist_ok=True)
    for f in ("mimicore_train.py", "onnx_export.py", "applio_patches.py"):
        urllib.request.urlretrieve(f"{REPO_RAW}/{f}", TOOLS / f)
    sh([sys.executable, str(TOOLS / "applio_patches.py"), str(APPLIO)])
    marker = HOME / ".installed"
    if not marker.exists():
        print("Installing (a few minutes)...")
        sh("pip install -q uv")
        sh(f"uv pip install --system -q -r requirements.txt onnx --extra-index-url https://download.pytorch.org/whl/cu128 "
           f"--index-strategy unsafe-best-match", cwd=APPLIO)
        print("Downloading voice analysis models...")
        sh([sys.executable, "core.py", "prerequisites", "--no-pretraineds-hifigan", "--models", "--no-exe"], cwd=APPLIO)
        titan = APPLIO / "rvc/models/pretraineds/custom/TITAN_Medium_recommended_for_speech"
        titan.mkdir(parents=True, exist_ok=True)
        for k in ("G", "D"):
            dst = titan / f"{k}-f048k-TITAN-Medium.pth"
            if not dst.exists():
                print(f"Downloading the base voice model ({k})...")
                urllib.request.urlretrieve(TITAN_URL.format(k), dst)
        marker.touch()
    return vram


def batch_for(vram):  # same rule as the desktop trainer
    return 16 if vram >= 20 else 12 if vram >= 11 else 8 if vram >= 7 else 4


def sync_resume(run_dir, resume, stop):
    """Every few minutes copy the latest checkpoint to Drive, so a dropped session can continue where it stopped."""
    seen = {}
    while not stop.wait(600):
        try:
            resume.mkdir(exist_ok=True)
            for p in list(run_dir.glob("*_2333333.pth")) + list(run_dir.glob("*e_*s.pth")) + [run_dir / "config.json"]:
                if p.exists() and seen.get(p.name) != p.stat().st_mtime:
                    shutil.copy2(p, resume / p.name)
                    seen[p.name] = p.stat().st_mtime
            if (run_dir / "eval").exists():
                shutil.copytree(run_dir / "eval", resume / "eval", dirs_exist_ok=True)
        except Exception as e:
            print(f"(checkpoint backup skipped: {e})")


def train(job, vram):
    spec = job.spec
    vid = re.sub(r"[^a-z0-9_-]+", "_", spec.get("id", "voice").lower())
    epochs = int(spec.get("epochs", 250))
    batch = int(spec.get("batch") or batch_for(vram))
    dataset, runs, out = WORK / "dataset", WORK / "runs", WORK / "out"
    shutil.rmtree(dataset, ignore_errors=True)
    dataset.mkdir(parents=True)
    recs = sorted((job.folder / "recordings").glob("*"))
    if not recs:
        raise SystemExit("The job has no recordings. Start the training again from Mimicore.")
    print(f"Copying {len(recs)} recording(s)...")
    for i, f in enumerate(recs):
        shutil.copy2(f, dataset / f"{i:03}{f.suffix.lower()}")
    run_dir = runs / vid
    resume = job.folder / "resume"
    if resume.exists() and any(resume.glob("*_2333333.pth")):
        print("Continuing from the last saved checkpoint...")
        shutil.copytree(resume, run_dir, dirs_exist_ok=True)
    stop = threading.Event()
    threading.Thread(target=sync_resume, args=(run_dir, resume, stop), daemon=True).start()
    job.write(state="training", step=1, steps=5, label="Cleaning & slicing audio", epoch=0, epochs=epochs, error=None)
    t0, e0 = None, 0
    p = subprocess.Popen([sys.executable, str(TOOLS / "mimicore_train.py"), "--name", vid, "--dataset", str(dataset),
                          "--runs", str(runs), "--out", str(out), "--epochs", str(epochs), "--batch", str(batch),
                          "--tools", str(TOOLS)], cwd=APPLIO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    last_write, tail = 0.0, []
    for line in p.stdout:
        line = line.rstrip()
        if job.cancelled():
            p.kill()
            stop.set()
            job.write(state="cancelled", label="Cancelled")
            raise SystemExit("Cancelled from Mimicore.")
        if line.startswith("@@"):
            v = json.loads(line[2:])
            if "step" in v:
                job.status.update(step=v["step"], steps=v["of"], label=v["label"])
                print(f"▶ Step {v['step']}/{v['of']}: {v['label']}")
            if "epoch" in v:
                e = v["epoch"]
                if t0 is None:
                    t0, e0 = time.time(), e
                elif e > e0:
                    job.status["eta_s"] = int((time.time() - t0) / (e - e0) * (epochs - e))
                job.status["epoch"] = e
                if e % 10 == 0:
                    eta = job.status.get("eta_s")
                    print(f"   epoch {e}/{epochs}" + (f" · about {eta // 60} min left" if eta else ""))
            if time.time() - last_write > 20 or "step" in v:
                job.write()
                last_write = time.time()
            continue
        tail = (tail + [line])[-40:]
    p.wait()
    stop.set()
    if p.returncode != 0:
        msg = next((l for l in reversed(tail) if l.strip()), "training failed")
        raise RuntimeError(msg)
    return out


def deliver(job, out):
    job.write(state="uploading", label="Sending the voice to your Drive")
    tmp = WORK / "voice.tar"
    with tarfile.open(tmp, "w") as t:
        for f in sorted(out.iterdir()):
            t.add(f, arcname=f.name)
    with open(job.folder / "voice.tar", "r+b") as dst, open(tmp, "rb") as src:  # in place: keeps the file id
        dst.seek(0)
        shutil.copyfileobj(src, dst, 8 << 20)
        dst.truncate()
        dst.flush()
        os.fsync(dst.fileno())
    if not job.spec.get("keep_files"):
        shutil.rmtree(job.folder / "recordings", ignore_errors=True)
        shutil.rmtree(job.folder / "resume", ignore_errors=True)
    job.write(state="done", label="Done", voice_bytes=tmp.stat().st_size)


job = None
try:
    job = find_job()
    who = job.spec.get("name", "your voice")
    q = job.spec.get("quality", "")
    banner(f"Training “{who}” {('· ' + q) if q else ''}. Keep this tab open.")
    vram = setup(job)
    out = train(job, vram)
    deliver(job, out)
    from google.colab import drive

    drive.flush_and_unmount()
    banner(f"Done! “{who}” is ready. Mimicore installs it automatically; you can close this tab.", "#10b981")
    time.sleep(5)
    from google.colab import runtime

    runtime.unassign()  # give the free GPU back
except SystemExit as e:
    if job and job.status.get("state") not in ("cancelled",):
        job.write(state="failed", error=str(e), label="Stopped")
    banner(str(e), "#ef4444")
except Exception as e:
    traceback.print_exc()
    msg = str(e).strip().splitlines()[-1] if str(e).strip() else type(e).__name__
    if "out of memory" in str(e).lower():
        msg = "The cloud GPU ran out of memory. Press ▶ again: training continues from the last checkpoint."
    if job:
        try:
            job.write(state="failed", error=msg[:500], label="Stopped")
        except Exception:
            pass
    banner(f"Training stopped: {msg}<br>Press ▶ again to continue from the last saved checkpoint.", "#ef4444")
